# [Navigating Graphs - RDFLib](https://rdflib.readthedocs.io/en/stable/intro_to_graphs/)

In [3]:
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import FOAF, RDF
from rdflib.term import Node

bob: URIRef = URIRef("http://example.org/people/bob")
alice: URIRef = URIRef("http://example.org/people/alice")

g: Graph = Graph()

g.add((bob, RDF.type, FOAF.Person))
g.add((bob, FOAF.name, Literal("Bob")))
g.add((alice, RDF.type, FOAF.Person))
g.add((alice, FOAF.name, Literal("Alice")))
g.add((alice, FOAF.knows, bob))

print(f"{len(g)} triples")

5 triples


## Graphs as Iterators

In [4]:
for s, p, o in g:
    if (s, p, o) not in g:
        raise RuntimeError("Iterator / Container Protocols are Broken!!")

print("Iteration OK")

Iteration OK


## Contains check

In [ ]:
if (bob, RDF.type, FOAF.Person) in g:
    print("This graph knows that Bob is a person!")

This graph knows that Bob is a person!


In [7]:
# ワイルドカード
if (bob, None, None) in g:
    print("This graph contains triples about Bob!")

This graph contains triples about Bob!


## Set Operations on RDFLib Graphs

In [13]:
G1: Graph = Graph()
G2: Graph = Graph()

G1.add((bob, RDF.type, FOAF.Person))
G1.add((bob, FOAF.name, Literal("Bob")))

G2.add((bob, RDF.type, FOAF.Person))
G2.add((alice, RDF.type, FOAF.Person))

def show_graph(label: str, graph: Graph) -> None:
    print(f"{label} ({len(graph)} triples):")
    for s, p, o in sorted(graph, key=lambda t: tuple(map(str, t))):
        print(f"  ({s}, {p}, {o})")

show_graph("G1", G1)
show_graph("G2", G2)

G1 (2 triples):
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)
G2 (2 triples):
  (http://example.org/people/alice, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)


In [14]:
# Union
union: Graph = G1 + G2
show_graph("G1 + G2", union)

# In-place union
inplace_union: Graph = G1 + Graph()
inplace_union += G2
show_graph("G1 += G2", inplace_union)

# Difference
difference: Graph = G1 - G2
show_graph("G1 - G2", difference)

# In-place difference
inplace_difference: Graph = G1 + Graph()
inplace_difference -= G2
show_graph("G1 -= G2", inplace_difference)

# Intersection
intersection: Graph = G1 & G2
show_graph("G1 & G2", intersection)

# Symmetric difference (XOR)
xor: Graph = G1 ^ G2
show_graph("G1 ^ G2", xor)


G1 + G2 (3 triples):
  (http://example.org/people/alice, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)
G1 += G2 (3 triples):
  (http://example.org/people/alice, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)
G1 - G2 (1 triples):
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)
G1 -= G2 (1 triples):
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)
G1 & G2 (1 triples):
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
G1 ^ G2 (2 triples):

## Basic Triple Matching

In [16]:
# Find all subjects of type foaf:Person
for s, p, o in g.triples((None, RDF.type, FOAF.Person)):
    print(f"{s} is a person")

http://example.org/people/bob is a person
http://example.org/people/alice is a person


In [17]:
# Find all subjects of any type
for s, p, o in g.triples((None, RDF.type, None)):
    print(f"{s} is a {o}")

http://example.org/people/bob is a http://xmlns.com/foaf/0.1/Person
http://example.org/people/alice is a http://xmlns.com/foaf/0.1/Person


In [19]:
# Create a graph containing Bob's triples
bobgraph: Graph = Graph()
bobgraph += g.triples((bob, None, None))

show_graph("Bob's triples", bobgraph)

Bob's triples (2 triples):
  (http://example.org/people/bob, http://www.w3.org/1999/02/22-rdf-syntax-ns#type, http://xmlns.com/foaf/0.1/Person)
  (http://example.org/people/bob, http://xmlns.com/foaf/0.1/name, Bob)


In [20]:
for person in g.subjects(RDF.type, FOAF.Person):
    print("{} is a person".format(person))


http://example.org/people/bob is a person
http://example.org/people/alice is a person


In [21]:
# Get a name of Bob
name: Node | None = g.value(bob, FOAF.name)
print("Bob's name:", name)

Bob's name: Bob


In [22]:
# Get the person who knows Bob
# Raise an exception if multiple values are found
person: Node | None = g.value(
    predicate=FOAF.knows,
    object=bob,
    any=False,
)
print("Person who knows Bob:", person)

Person who knows Bob: http://example.org/people/alice


補足：value() は、subject と object のどちらが None なのかによって、取得する要素を切り替える